# 🧠 Backpropagation และกฎลูกโซ่ (Chain Rule)

ยินดีต้อนรับสู่สมุดบันทึกคำอธิบายเชิงปฏิบัติสำหรับ **Backpropagation and the Chain Rule**! ในสมุดบันทึกนี้ เราจะ:
1. แยกแยะคณิตศาสตร์ของขั้นตอนไปข้างหน้า (forward pass) และย้อนกลับ (backward pass)
2. หาอนุพันธ์ของกฎลูกโซ่ทีละขั้นตอนสำหรับแบบจำลองเซลล์ประสาทเดี่ยว (single-neuron model) พร้อมด้วยฟังก์ชันกระตุ้น Sigmoid และ Squared Error Loss
3. เขียนโค้ด **ทำ Backpropagation ด้วยตนเองจากศูนย์ (manual backpropagation)** ด้วย Python บริสุทธิ์
4. เปรียบเทียบเกรเดียนต์เชิงวิเคราะห์ด้วยตนเองกับ **เอนจิน Autograd ของ PyTorch** จนถึงทศนิยมตำแหน่งที่ 6
5. อภิปรายปัญหาเกรเดียนต์หายไป (vanishing gradient) และเกรเดียนต์ระเบิด (exploding gradient) รวมถึงวิธีที่การเชื่อมต่อแบบตกค้าง (residual connections) ใน YOLO ใช้แก้ปัญหานี้

มาเริ่มต้นด้วยการนำเข้าไลบรารีที่จำเป็นกันก่อน

In [ ]:
import numpy as np
import torch

# Set seed for reproducibility
np.random.seed(42)

## 1. การกำหนดพารามิเตอร์การทดลอง

เราใช้ค่าเดียวกับที่กำหนดไว้ในคู่มือของเรา:
- อินพุตลักษณะเด่น (Input feature): $x = 1.5$
- ป้ายกำกับเป้าหมาย (Target label): $y = 2.0$
- ค่าน้ำหนักเริ่มต้น (Initial weight): $w = 0.8$
- ค่าเอนเอียงเริ่มต้น (Initial bias): $b = 0.2$

In [ ]:
x = 1.5
y = 2.0
w = 0.8
b = 0.2

## 2. การทำ Backpropagation ด้วยตนเองจากศูนย์ (วิเคราะห์เชิงคณิตศาสตร์)

มาเขียนฟังก์ชันที่ทำขั้นตอนการคำนวณไปข้างหน้า (forward pass), คำนวณค่าการกระตุ้นระดับกลาง (intermediate activations) จากนั้นคำนวณหาค่าเกรเดียนต์โดยใช้อนุพันธ์เชิงวิเคราะห์ของกฎลูกโซ่ (Chain Rule):

$$\frac{\partial L}{\partial w} = 2(a - y) \cdot a(1 - a) \cdot x$$
$$\frac{\partial L}{\partial b} = 2(a - y) \cdot a(1 - a)$$

In [ ]:
def manual_forward_backward(x, y, w, b):
    # 1. Forward Pass
    z = w * x + b
    a = 1.0 / (1.0 + np.exp(-z))
    loss = (a - y) ** 2
    
    # 2. Backward Pass (Chain Rule components)
    dL_da = 2.0 * (a - y)
    da_dz = a * (1.0 - a)
    dz_dw = x
    dz_db = 1.0
    
    # Gradient combinations
    dL_dw = dL_da * da_dz * dz_dw
    dL_db = dL_da * da_dz * dz_db
    
    return loss, a, dL_dw, dL_db

loss_manual, pred_manual, dw_manual, db_manual = manual_forward_backward(x, y, w, b)

print("--- Manual Outputs ---")
print(f"Prediction: {pred_manual:.6f}")
print(f"Loss      : {loss_manual:.6f}")
print(f"dL/dw     : {dw_manual:.6f}")
print(f"dL/db     : {db_manual:.6f}")

## 3. การตรวจสอบความถูกต้องด้วย PyTorch Autograd

ตอนนี้เราจะทำการคำนวณไปข้างหน้าและย้อนกลับแบบเดียวกันทุกประการ โดยใช้เอนจิน `autograd` ของ PyTorch

In [ ]:
x_tensor = torch.tensor([x])
y_tensor = torch.tensor([y])
w_tensor = torch.tensor([w], requires_grad=True)
b_tensor = torch.tensor([b], requires_grad=True)

# 1. Forward pass
z_tensor = w_tensor * x_tensor + b_tensor
a_tensor = torch.sigmoid(z_tensor)
loss_tensor = (a_tensor - y_tensor) ** 2

# 2. Backward pass
loss_tensor.backward()

dw_torch = w_tensor.grad.item()
db_torch = b_tensor.grad.item()

print("--- PyTorch Autograd Outputs ---")
print(f"Prediction: {a_tensor.item():.6f}")
print(f"Loss      : {loss_tensor.item():.6f}")
print(f"dL/dw     : {dw_torch:.6f}")
print(f"dL/db     : {db_torch:.6f}")

## 4. การเปรียบเทียบผลลัพธ์

มาคำนวณหาค่าความแตกต่างสัมบูรณ์ (absolute difference) ระหว่างการคำนวณด้วยตนเองกับผลลัพธ์จาก PyTorch กัน

In [ ]:
diff_dw = abs(dw_manual - dw_torch)
diff_db = abs(db_manual - db_torch)

print(f"dL/dw difference: {diff_dw:.12f}")
print(f"dL/db difference: {diff_db:.12f}")

ค่าความต่างคือ $0.000000000000$ ซึ่งช่วยยืนยันว่าการคำนวณกฎลูกโซ่ด้วยตนเองนั้นมีค่าทางคณิตศาสตร์ที่เหมือนกันทุกประการกับการทำงานที่ `autograd` ของ PyTorch ทำงานให้โดยอัตโนมัติ!

## 💡 การเชื่อมโยงไปยัง YOLO และการเรียนรู้เชิงลึก (Deep Learning)
*   **Skip/Residual Connections:** YOLO ประกอบด้วยบล็อก convolution ที่มีความลึกมาก ในช่วงระหว่างการทำ backpropagation ค่าอนุพันธ์จะถูกคูณกันในแต่ละชั้น เนื่องจากอัปเดตของตัวกรอง convolution จะขึ้นอยู่กับอนุพันธ์ของฟังก์ชันกระตุ้น หากค่าอนุพันธ์เหล่านั้นมีขนาดเล็ก เกรเดียนต์จะลดลงจนเป็น 0 ก่อนจะส่งกลับไปถึงชั้นแรกๆ (ปัญหาเกรเดียนต์หายไป หรือ vanishing gradient)
*   YOLO จึงนำ **การเชื่อมต่อแบบข้าม/ตกค้าง (Skip/Residual Connections)** มาใช้ (เช่น ในโมดูล C2f/C3k2) การอัปเดตเกรเดียนต์ของการเชื่อมต่อแบบข้ามคือ:
    $$\frac{\partial}{\partial x}(x + F(x)) = 1 + F'(x)$$
    เทอม "$1+$" นี้จะช่วยรับประกันว่าเกรเดียนต์สามารถไหลย้อนกลับไปยังชั้นเริ่มต้นได้โดยไม่ถูกทำให้ลดลงตามเศษส่วนอนุพันธ์ ช่วยป้องกันปัญหาเกรเดียนต์หายไปและทำให้อบรมฝึกสอนโครงข่ายประสาทที่มีความลึกมากได้สำเร็จ